# RAG mit der Siemens-Anleitung
Dieses Notebook nutzt dieselben Funktionen wie die Web-App. Kernel: Python 3.12 aus der Projektumgebung. Im Repository-Stamm starten. Modelle werden beim ersten Lauf heruntergeladen. Es erfolgen hier keine kostenpflichtigen Antwort-API-Aufrufe.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'rag_engine.py').exists() and (ROOT.parent / 'rag_engine.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'rag_engine.py').exists(), 'Notebook im geklonten Projekt öffnen'
sys.path.insert(0, str(ROOT))
import rag_engine


## 1. Dokument und Chunks
Die aktive Markdown-Datei stammt aus der PDF-Aufbereitung. Tabellenzeilen und Tokenlimits werden im gemeinsamen Kern behandelt. Der erste Aufruf lädt E5.


In [ ]:
model = rag_engine.get_embed_model()
nodes = rag_engine.prepare_nodes(ROOT / 'siemens_wissen.md', tokenizer=model._model.tokenizer.encode)
print('Chunks:', len(nodes))
print(nodes[0].get_content()[:800])


## 2. Hybrid-Suche und Reranking
Der BGE-Reranker benötigt zusätzliche Modellgewichte und Arbeitsspeicher. Scores messen Relevanz; sie sind keine Wahrheitswahrscheinlichkeit.


In [ ]:
index = rag_engine.build_or_load_index()
retriever = rag_engine.make_retriever(index)
question = 'Was bedeutet E:18?'
hits = retriever.retrieve(question)
reranker = rag_engine.get_reranker()
if reranker is not None:
    hits = reranker.postprocess_nodes(hits, query_str=question)
for hit in hits[:rag_engine.FINAL_K]:
    print(hit.score, hit.node.get_content()[:700], '\n')


## 3. Prüffragen
`python eval/run_eval.py --output docs/evaluation/hybrid-rerank.json` misst Stichworttreffer für zehn Fragen. Für Antwortqualität müssen zusätzlich echte Modellantworten gegen die Originalquelle geprüft werden. Die GUI wird mit `python server.py` gestartet und erlaubt den Wechsel zwischen LM Studio und OpenAI.
Siehe `docs/TECHNICAL.md`, `docs/AUDIT.md` und den PowerPoint-Foliensatz.
